In [ ]:
"""
ML4SCI 2026 — Specific Test IV: Neural Operator Classifier
Strong Gravitational Lensing: No Substructure / Subhalo / Vortex
Author  : <your name>
Backbone: 2-D Fourier Neural Operator (FNO2d) + classification head
Baseline: 99.1 % accuracy (CNN, Common Test I)
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
import os, random, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False   # set True for speed if input size fixed

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASS = 3
CLASS_NAMES = ["no_sub", "sphere", "vortex"]     # adjust to your folder names

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATASET
# ─────────────────────────────────────────────────────────────────────────────
class LensingDataset(Dataset):
    """
    Expects directory layout:
        dataset/
          train/
            no_sub/    *.npy  or  *.png
            sphere/
            vortex/
          val/   (optional – we split from train if absent)
    Supports both .npy arrays and common image formats.
    """
    def __init__(self, root: str, split: str = "train",
                 img_size: int = 64, augment: bool = False):
        self.samples   = []
        self.augment   = augment
        self.img_size  = img_size

        root_path = Path(root) / split
        if not root_path.exists():
            raise FileNotFoundError(f"Split folder not found: {root_path}")

        for label, cls in enumerate(CLASS_NAMES):
            cls_dir = root_path / cls
            if not cls_dir.exists():
                print(f"[WARN] class folder missing: {cls_dir}")
                continue
            for fp in cls_dir.glob("*"):
                if fp.suffix.lower() in (".npy", ".png", ".jpg", ".jpeg", ".fits"):
                    self.samples.append((str(fp), label))

        print(f"[{split}] {len(self.samples)} samples")

        # ── augmentation pipeline ─────────────────────────────────────────
        if augment:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(15),
                transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.CenterCrop(img_size),
            ])

    def _load(self, path: str) -> torch.Tensor:
        """Return (1, H, W) float32 tensor in [0,1]."""
        if path.endswith(".npy"):
            arr = np.load(path).astype(np.float32)
            if arr.ndim == 2:
                arr = arr[None]          # add channel dim
            elif arr.ndim == 3:
                arr = arr[:1]            # keep first channel only
        else:
            img = Image.open(path).convert("L")
            arr = np.array(img, dtype=np.float32)[None] / 255.0

        t = torch.from_numpy(arr)
        # resize to uniform spatial size
        t = F.interpolate(t.unsqueeze(0), size=self.img_size,
                          mode="bilinear", align_corners=False).squeeze(0)
        return t

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self._load(path)
        # apply PIL-based augmentations on uint8 equivalent
        if self.augment:
            pil = transforms.ToPILImage()(img)
            pil = self.transform(pil)
            img = transforms.ToTensor()(pil)
        return img, label


def build_loaders(data_root: str, img_size: int = 64,
                  batch_size: int = 64, val_frac: float = 0.15):
    """Build train / val / test loaders."""
    train_ds_full = LensingDataset(data_root, "train",
                                   img_size=img_size, augment=True)

    n_val   = max(1, int(len(train_ds_full) * val_frac))
    n_train = len(train_ds_full) - n_val
    train_ds, val_ds = random_split(train_ds_full, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))

    # test set (no augment)
    test_path = Path(data_root) / "test"
    if test_path.exists():
        test_ds = LensingDataset(data_root, "test", img_size=img_size, augment=False)
    else:
        test_ds = val_ds   # fallback

    kw = dict(num_workers=4, pin_memory=True, persistent_workers=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)

    return train_loader, val_loader, test_loader


# ─────────────────────────────────────────────────────────────────────────────
# 2. FOURIER NEURAL OPERATOR — core building blocks
# ─────────────────────────────────────────────────────────────────────────────

class SpectralConv2d(nn.Module):
    """
    2-D Fourier layer  (FNO-style)
    ────────────────────────────────────────────────────────────────────────
    Forward pass:
      1. Real FFT2  →  complex spectrum  [B, C_in, H, W//2+1]
      2. Truncate to (modes1 × modes2) lowest frequencies in each dimension
      3. Multiply by learned complex weights  R ∈ ℂ^{C_in × C_out × modes1 × modes2}
      4. Pad back to full spectrum size
      5. Inverse real FFT2  →  spatial field  [B, C_out, H, W]

    Key difference from a standard conv:
      • Weight sharing is in Fourier (frequency) space, not pixel space
      • Each weight couples a global mode — effectively infinite receptive field
      • Operates on *functions*, resolution-independent up to Nyquist
    """
    def __init__(self, in_channels: int, out_channels: int,
                 modes1: int, modes2: int):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1   # kept Fourier modes along dim -2
        self.modes2 = modes2   # kept Fourier modes along dim -1

        scale = 1.0 / (in_channels * out_channels)
        # store real & imag separately for full torch.compile / ONNX compat
        self.wr1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))
        self.wi1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))
        self.wr2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))
        self.wi2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))

    # ── helpers ─────────────────────────────────────────────────────────────
    def _compl_mul2d(self, x_r, x_i, w_r, w_i):
        """Batched complex einsum:  out = x * w,  shapes (B,Ci,M1,M2)"""
        o_r = torch.einsum("bixy,ioxy->boxy", x_r, w_r) \
            - torch.einsum("bixy,ioxy->boxy", x_i, w_i)
        o_i = torch.einsum("bixy,ioxy->boxy", x_r, w_i) \
            + torch.einsum("bixy,ioxy->boxy", x_i, w_r)
        return o_r, o_i

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        # rfft2 → complex tensor
        x_ft = torch.fft.rfft2(x, norm="ortho")           # (B, C, H, W//2+1)
        x_ft_r, x_ft_i = x_ft.real, x_ft.imag

        out_r = torch.zeros(B, self.out_channels, H, W // 2 + 1, device=x.device)
        out_i = torch.zeros_like(out_r)

        # lower-left block (positive frequencies)
        r1, i1 = self._compl_mul2d(
            x_ft_r[:, :, :self.modes1, :self.modes2],
            x_ft_i[:, :, :self.modes1, :self.modes2],
            self.wr1, self.wi1)
        out_r[:, :, :self.modes1, :self.modes2] = r1
        out_i[:, :, :self.modes1, :self.modes2] = i1

        # upper-left block (negative frequencies / conjugate)
        r2, i2 = self._compl_mul2d(
            x_ft_r[:, :, -self.modes1:, :self.modes2],
            x_ft_i[:, :, -self.modes1:, :self.modes2],
            self.wr2, self.wi2)
        out_r[:, :, -self.modes1:, :self.modes2] = r2
        out_i[:, :, -self.modes1:, :self.modes2] = i2

        out_ft = torch.complex(out_r, out_i)
        return torch.fft.irfft2(out_ft, s=(H, W), norm="ortho")  # (B, C_out, H, W)


class FNOBlock2d(nn.Module):
    """
    One FNO residual block:
        x  →  SpectralConv2d(x)  +  Conv1×1(x)  →  activation
    The local bypass (1×1 conv) preserves fine-grained spatial detail that
    high-frequency truncation discards.
    """
    def __init__(self, channels: int, modes1: int, modes2: int):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes1, modes2)
        self.bypass   = nn.Conv2d(channels, channels, 1)   # point-wise mix
        self.norm     = nn.InstanceNorm2d(channels, affine=True)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.bypass(x)))


# ─────────────────────────────────────────────────────────────────────────────
# 3. FULL CLASSIFIER
# ─────────────────────────────────────────────────────────────────────────────

class FNOClassifier(nn.Module):
    """
    Architecture
    ────────────────────────────────────────────────────────────────────────
    Input  : (B, 1, H, W)  — single-channel lensing image
    ↓
    Lifting layer   : Conv2d(1 → width)           — lift to latent channels
    ↓  × depth
    FNO block       : SpectralConv + bypass conv  — operate in function space
    ↓
    Projection      : Conv2d(width → proj_ch)     — compress channels
    ↓
    Adaptive avg + max pool → concatenate
    ↓
    MLP classifier  : FC → BN → GELU → Dropout → FC(num_classes)
    ────────────────────────────────────────────────────────────────────────
    """
    def __init__(self,
                 in_channels : int = 1,
                 width       : int = 64,    # latent channel width
                 modes1      : int = 16,    # Fourier modes to keep (dim -2)
                 modes2      : int = 16,    # Fourier modes to keep (dim -1)
                 depth       : int = 4,     # number of FNO blocks
                 num_classes : int = 3,
                 dropout     : float = 0.3):
        super().__init__()

        # 1. lift input to width channels
        self.lift = nn.Sequential(
            nn.Conv2d(in_channels, width, 3, padding=1),
            nn.InstanceNorm2d(width, affine=True),
            nn.GELU(),
        )

        # 2. FNO blocks
        self.fno_blocks = nn.Sequential(
            *[FNOBlock2d(width, modes1, modes2) for _ in range(depth)]
        )

        # 3. channel projection
        proj_ch = width * 2
        self.project = nn.Sequential(
            nn.Conv2d(width, proj_ch, 1),
            nn.BatchNorm2d(proj_ch),
            nn.GELU(),
        )

        # 4. dual pooling → flattened vector of size 2 * proj_ch
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)

        # 5. classification head
        feat_dim = proj_ch * 2
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.lift(x)
        x = self.fno_blocks(x)
        x = self.project(x)
        x = torch.cat([self.gap(x), self.gmp(x)], dim=1)
        return self.head(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ─────────────────────────────────────────────────────────────────────────────
# 4. TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n   = logits.size(-1)
        log = F.log_softmax(logits, dim=-1)
        nll = F.nll_loss(log, targets)
        smooth = -log.mean(dim=-1).mean()
        return (1 - self.smoothing) * nll + self.smoothing / n * smooth * n


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device.type):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_logits.append(logits.cpu())
        all_labels.append(labels)
    logits = torch.cat(all_logits);  labels = torch.cat(all_labels)
    probs  = torch.softmax(logits, dim=1).numpy()
    preds  = logits.argmax(1).numpy()
    labs   = labels.numpy()
    acc    = (preds == labs).mean()
    return acc, probs, preds, labs


# ─────────────────────────────────────────────────────────────────────────────
# 5. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train(model, train_loader, val_loader,
          epochs=50, lr=2e-3, weight_decay=1e-4,
          device=DEVICE, save_path="best_fno.pt"):

    model.to(device)
    criterion = LabelSmoothingCE(0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=weight_decay)
    # OneCycleLR: warms up then cosine-decays — ideal for FNOs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=len(train_loader), pct_start=0.1,
        anneal_strategy="cos", div_factor=10.0, final_div_factor=1e3)

    scaler    = torch.amp.GradScaler()
    best_val  = 0.0
    history   = {"train_loss": [], "train_acc": [], "val_acc": []}

    print(f"\nParameters : {model.count_parameters():,}")
    print(f"Epochs     : {epochs},  LR: {lr},  Device: {device}\n")
    print(f"{'Epoch':>6}  {'T-Loss':>8}  {'T-Acc':>7}  {'V-Acc':>7}  {'LR':>9}")
    print("─" * 50)

    for ep in range(1, epochs + 1):
        t0 = time.time()
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer,
                                        criterion, scaler, device)
        v_acc, *_ = evaluate(model, val_loader, device)
        scheduler.step()    # already called per step inside, but safe here too

        history["train_loss"].append(t_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)

        if v_acc > best_val:
            best_val = v_acc
            torch.save({"epoch": ep, "state_dict": model.state_dict(),
                        "val_acc": best_val}, save_path)

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"{ep:>6}  {t_loss:>8.4f}  {t_acc:>6.2%}  {v_acc:>6.2%}  {lr_now:>9.6f}"
              f"  {'✓' if v_acc == best_val else ''}"
              f"  [{time.time()-t0:.1f}s]")

    print(f"\nBest validation accuracy: {best_val:.4%}")
    return history, best_val


# ─────────────────────────────────────────────────────────────────────────────
# 6. EVALUATION & PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_roc(probs, labels, class_names=CLASS_NAMES, save="roc_curve.png"):
    """
    One-vs-Rest ROC curves for each class + macro-average AUC.
    """
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(labels, classes=list(range(len(class_names))))

    fig, ax = plt.subplots(figsize=(8, 6))
    colors  = ["#4C72B0", "#DD8452", "#55A868"]
    aucs    = []

    for i, (cls, col) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc     = auc(fpr, tpr)
        aucs.append(roc_auc)
        ax.plot(fpr, tpr, color=col, lw=2,
                label=f"{cls}  (AUC = {roc_auc:.4f})")

    macro_auc = np.mean(aucs)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel("False Positive Rate", fontsize=13)
    ax.set_ylabel("True Positive Rate",  fontsize=13)
    ax.set_title(f"ROC — FNO Classifier   (Macro AUC = {macro_auc:.4f})", fontsize=14)
    ax.legend(loc="lower right", fontsize=11)
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.close()
    print(f"ROC saved → {save}")
    print(f"Per-class AUC : {dict(zip(class_names, [f'{a:.4f}' for a in aucs]))}")
    print(f"Macro AUC     : {macro_auc:.4f}")
    return macro_auc


def plot_confusion(preds, labels, class_names=CLASS_NAMES, save="confusion.png"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — FNO Classifier")
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.close()
    print(f"Confusion matrix saved → {save}")


def plot_history(history, save="training_history.png"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ep = range(1, len(history["train_loss"]) + 1)
    ax1.plot(ep, history["train_loss"], label="Train loss")
    ax1.set(title="Loss", xlabel="Epoch", ylabel="Label-smoothed CE")
    ax1.legend()
    ax2.plot(ep, history["train_acc"], label="Train acc")
    ax2.plot(ep, history["val_acc"],   label="Val acc")
    ax2.set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
    ax2.legend()
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.close()
    print(f"History saved → {save}")


# ─────────────────────────────────────────────────────────────────────────────
# 7. SPECTRAL ANALYSIS VISUALIZER (Bonus — shows what FNO "sees")
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def visualize_spectral_features(model, loader, device=DEVICE, save="spectral.png"):
    """
    Plots the magnitude of the first spectral conv weight (frequency heatmap)
    alongside a sample image and its Fourier spectrum — great for your report.
    """
    model.eval()
    imgs, labels = next(iter(loader))
    img = imgs[0].to(device)                          # (1, H, W)

    # Fourier spectrum of the input
    ft   = torch.fft.rfft2(img[0], norm="ortho")
    mag  = torch.log1p(ft.abs()).cpu().numpy()

    # First spectral layer learned weights
    first_spectral = model.fno_blocks[0].spectral
    w_mag = torch.sqrt(first_spectral.wr1**2 + first_spectral.wi1**2)
    w_avg = w_mag.mean(dim=(0, 1)).cpu().numpy()     # (modes1, modes2)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(img[0].cpu(), cmap="inferno")
    axes[0].set_title(f"Input image  (class={CLASS_NAMES[labels[0]]})")

    axes[1].imshow(mag, cmap="plasma")
    axes[1].set_title("log|FFT| of input")

    im = axes[2].imshow(w_avg, cmap="viridis", aspect="auto")
    axes[2].set_title("FNO weight magnitude in Fourier space")
    plt.colorbar(im, ax=axes[2])

    for ax in axes: ax.axis("off") if ax != axes[2] else None
    plt.suptitle("What the Fourier Neural Operator operates on", fontsize=13)
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.close()
    print(f"Spectral viz saved → {save}")


# ─────────────────────────────────────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="FNO Lensing Classifier")
    parser.add_argument("--data",       default="dataset",  help="path to dataset root")
    parser.add_argument("--img_size",   type=int,  default=64)
    parser.add_argument("--batch_size", type=int,  default=64)
    parser.add_argument("--epochs",     type=int,  default=60)
    parser.add_argument("--lr",         type=float,default=3e-3)
    parser.add_argument("--width",      type=int,  default=64,
                        help="FNO latent channel width")
    parser.add_argument("--modes",      type=int,  default=16,
                        help="Number of Fourier modes to keep per dimension")
    parser.add_argument("--depth",      type=int,  default=4,
                        help="Number of FNO blocks")
    parser.add_argument("--dropout",    type=float,default=0.3)
    parser.add_argument("--resume",     default=None, help="checkpoint to resume from")
    parser.add_argument("--eval_only",  action="store_true")
    args = parser.parse_args()

    # ── data ─────────────────────────────────────────────────────────────────
    train_loader, val_loader, test_loader = build_loaders(
        args.data, img_size=args.img_size, batch_size=args.batch_size)

    # ── model ─────────────────────────────────────────────────────────────────
    model = FNOClassifier(
        in_channels=1,
        width=args.width,
        modes1=args.modes,
        modes2=args.modes,
        depth=args.depth,
        num_classes=NUM_CLASS,
        dropout=args.dropout,
    )

    if args.resume:
        ckpt = torch.load(args.resume, map_location="cpu")
        model.load_state_dict(ckpt["state_dict"])
        print(f"Resumed from {args.resume}  (epoch {ckpt['epoch']}, "
              f"val_acc {ckpt['val_acc']:.4%})")

    # ── train ─────────────────────────────────────────────────────────────────
    if not args.eval_only:
        history, best_val = train(
            model, train_loader, val_loader,
            epochs=args.epochs, lr=args.lr,
            device=DEVICE, save_path="best_fno.pt")
        plot_history(history)

    # ── final evaluation on test set ──────────────────────────────────────────
    ckpt = torch.load("best_fno.pt", map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    model.to(DEVICE)

    test_acc, probs, preds, labels = evaluate(model, test_loader, DEVICE)
    print(f"\n{'═'*50}")
    print(f"Test Accuracy : {test_acc:.4%}")
    print(f"{'═'*50}")
    print(classification_report(labels, preds, target_names=CLASS_NAMES))

    macro_auc = plot_roc(probs, labels)
    plot_confusion(preds, labels)
    visualize_spectral_features(model, test_loader)

    print("\n── Summary ──────────────────────────────────────────")
    print(f"  Model parameters : {model.count_parameters():,}")
    print(f"  Test accuracy    : {test_acc:.4%}")
    print(f"  Macro AUC        : {macro_auc:.4f}")
    print(f"  Baseline acc     : 99.10%  (CNN, Common Test I)")
    print("─────────────────────────────────────────────────────")